# CNN MNIST accelerator - PYNQ-Z2 hardware test
Run this notebook **on the PYNQ-Z2 board** from a clone of the repository (paths below are relative to `deploy/pynq_z2/`).

> Status: written from the original notebook but **not yet executed on a board** in this release (no board was available).
> Expected results are derived from the RTL simulation log and the bit-accurate golden model in `tools/golden_model.py`:
> the 1000 test vectors should give **900/1000 correct** and **1000/1000 decisions identical** to the simulation.

## 1. Program the bitstream

In [ ]:
import numpy as np
from pynq import Overlay, allocate

overlay = Overlay('cnn_mnist.bit')          # cnn_mnist.hwh must be in the same folder
dma_send = overlay.axi_dma_0.sendchannel
dma_recv = overlay.axi_dma_0.recvchannel

input_buffer = allocate(shape=(784,), dtype=np.uint8)
output_buffer = allocate(shape=(1,), dtype=np.uint8)

def classify(img_u8):
    """img_u8: 28x28 uint8 array (0..255) -> digit predicted by the FPGA."""
    input_buffer[:] = np.asarray(img_u8, dtype=np.uint8).reshape(784)
    dma_recv.transfer(output_buffer)
    dma_send.transfer(input_buffer)
    dma_send.wait()
    dma_recv.wait()
    return int(output_buffer[0])

## 2. Single bitmap image (`train_0.bmp`, expected digit 5)

In [ ]:
from PIL import Image
from matplotlib import pyplot

img = np.array(Image.open('../../model/bmp/train_0.bmp'))
pyplot.imshow(img, cmap='gray')
print('Predicted output:', classify(img))

## 3. Batch verification against the RTL simulation
Uses the same 1000 images as `axis_cnn_mnist_1000_tb` (labels are `j % 10`).

In [ ]:
import re

vec = np.array([int(t, 16) for t in open('../../sim/testvector/input_1000.txt').read().split()],
               dtype=np.uint8).reshape(1000, 28, 28)
sim = {}
for line in open('../../sim/results/simulate_1000_axis_cnn_mnist_1000_tb.log'):
    m = re.match(r'Input image (\d+): original value = \d+, decision = (\d+)', line)
    if m:
        sim[int(m[1])] = int(m[2])

hw = np.array([classify(vec[j]) for j in range(1000)])
labels = np.arange(1000) % 10
same = sum(hw[j] == sim[j] for j in range(1000))
print('HW accuracy            : %d/1000 (expected 900)' % (hw == labels).sum())
print('HW == RTL simulation   : %d/1000 (expected 1000)' % same)

## 4. Timing (single image)
The PL clock of this design is 50 MHz; the core needs ~1281 cycles per image (~25.6 us). The figure measured here also
includes the DMA driver overhead on the ARM side.

In [ ]:
from time import perf_counter

N = 200
t0 = perf_counter()
for _ in range(N):
    classify(vec[0])
print('mean latency per image: %.3f ms' % ((perf_counter() - t0) / N * 1000))